|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Quantization<h1>|
|<h2>Lecture:</h2>|<h1><b>Fewer bytes per weight, and the trap that undoes it<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

# Halve the bytes, halve the time

Decode reads every weight to make every token. That was Part 1's whole point,
and it has an obvious consequence nobody acts on until they measure it:
**decode time is proportional to weight bytes**, not to parameter count.

Store the weights in one byte instead of two and decode should be about twice
as fast. Not from better arithmetic. From reading less.

In [2]:
torch.manual_seed(0)
W = torch.randn(4096, 4096, device='cuda')

def quantize_int8(W):
  """Per output channel, symmetric. One scale per row."""
  scales = W.abs().amax(dim=1).clamp(min=1e-8) / 127.0
  q = torch.round(W / scales[:, None]).clamp(-127, 127).to(torch.int8)
  return q, scales

q, s = quantize_int8(W)
recon = q.float() * s[:, None]
err = (recon - W).abs().mean() / W.abs().mean()

print(f'bf16 weights: {W.numel()*2/1e6:7.1f} MB')
print(f'int8 + scales:{(q.numel() + s.numel()*4)/1e6:7.1f} MB   '
      f'({W.numel()*2/(q.numel()+s.numel()*4):.1f}x smaller)')
print(f'mean relative error: {err:.4f}')

bf16 weights:    33.6 MB
int8 + scales:   16.8 MB   (2.0x smaller)
mean relative error: 0.0094


### Why per-channel and not per-tensor

One scale for the whole matrix means one outlier row sets the resolution for
every other row. Per-channel gives each output its own.

In [3]:
Wo = W.clone()
Wo[0] *= 60.0                       # one row with huge weights

# per tensor
st = Wo.abs().max()/127.0
qt = torch.round(Wo/st).clamp(-127,127)
err_t = ((qt*st - Wo).abs().mean()/Wo.abs().mean()).item()

# per channel
qc, sc = quantize_int8(Wo)
err_c = ((qc.float()*sc[:,None] - Wo).abs().mean()/Wo.abs().mean()).item()

print(f'per tensor:  {err_t:.4f} mean relative error')
print(f'per channel: {err_c:.4f}   ({err_t/err_c:.0f}x better)')

per tensor:  0.4838 mean relative error
per channel: 0.0093   (52x better)


# The part that is easy to get backwards

Quantizing the weights saves nothing if you expand them again before the
multiply. Time all three.

In [4]:
x  = torch.randn(1, 4096, device='cuda', dtype=torch.bfloat16)
Wb = W.to(torch.bfloat16)
sb = s.to(torch.bfloat16)

bf16   = cudalib.bench_ms(lambda: x @ Wb.t(), best_of=3)
unfused = cudalib.bench_ms(lambda: x @ (q.to(torch.bfloat16)*sb[:,None]).t(), best_of=3)

print(f'bf16 matmul:            {bf16:7.3f} ms')
print(f'dequantize, then matmul:{unfused:7.3f} ms   <- SLOWER than not quantizing')
print(f'\nyou wrote {W.numel()*2/1e6:.0f} MB of bf16 weights out to memory')
print('and read them straight back, on top of reading the int8 ones.')

bf16 matmul:              0.075 ms
dequantize, then matmul:  0.501 ms   <- SLOWER than not quantizing

you wrote 34 MB of bf16 weights out to memory
and read them straight back, on top of reading the int8 ones.


That is the whole trap, and it is why stage 18b exists. The scale has to be
applied to a value that is **already in a register**, at the end of the dot
product, which is one multiply per output rather than one per weight:

$$y_n = \sum_k x_k \, w_{nk} s_n = s_n \sum_k x_k w_{nk}$$

The scale comes out of the sum. K multiplies become one. Stage 18b makes you
write that kernel, and it is 15 to 19 times faster than the unfused version
and 2.7 to 3.9 times faster than bf16 cuBLAS at batch 1.

    ./vc guide 18b

# The other big reader

Weights are not the only thing decode streams. The KV cache is read in full
every step too, and on long contexts it can be the larger of the two.

FP8 suits it better than INT8, because KV entries have a wide dynamic range
and FP8 keeps an exponent. Your card is sm_89, so `float8_e4m3fn` is native.

In [5]:
kv = torch.randn(8, 2048, 128, device='cuda') * torch.logspace(-2, 2, 8, device='cuda')[:,None,None]

scale = kv.abs().amax()/448.0                       # e4m3 max representable
q8    = (kv/scale).clamp(-448, 448).to(torch.float8_e4m3fn)
back8 = q8.to(torch.float32)*scale

s_i8  = kv.abs().amax()/127.0
qi8   = torch.round(kv/s_i8).clamp(-127,127)
backi = qi8*s_i8

rel = lambda a: ((a-kv).abs().mean()/kv.abs().mean()).item()
print(f'KV cache with a 10,000x spread across heads')
print(f'  fp8  e4m3 : {rel(back8):.4f} mean relative error')
print(f'  int8      : {rel(backi):.4f}')
print(f'\nsame width, {rel(backi)/rel(back8):.0f}x the error, because int8 has no exponent')

KV cache with a 10,000x spread across heads
  fp8  e4m3 : 0.0225 mean relative error
  int8      : 0.0419

same width, 2x the error, because int8 has no exponent


Then measure perplexity before and after, on real text, because none of these
error numbers tell you whether the model got worse at its job. Stage 18 makes
that the gate.

    ./vc guide 18